<a href="https://colab.research.google.com/github/aditya01-dev/Natural-Language-Processing-Lab/blob/main/NLP_Practical_32_Hybrid_Sequence_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Aditya kumar Singh
# Practical 32: Hybrid Probabilistic and FSM Sequence Modeling
# Objective: Combine Markov probabilistic prediction with FSM constraints.

# Import required libraries
import random
from collections import defaultdict

# Define states
states = ["START", "GREETING", "QUESTION", "ANSWER", "END"]

# Define valid FSM transitions
fsm_transitions = {
    "START": ["GREETING"],
    "GREETING": ["QUESTION", "END"],
    "QUESTION": ["ANSWER"],
    "ANSWER": ["QUESTION", "END"],
    "END": []
}

# Training sequences
training_sequences = [
    ["START", "GREETING", "QUESTION", "ANSWER", "END"],
    ["START", "GREETING", "QUESTION", "ANSWER", "QUESTION", "ANSWER", "END"],
    ["START", "GREETING", "END"],
    ["START", "GREETING", "QUESTION", "ANSWER", "END"]
]

# Learn transition frequencies
transition_counts = defaultdict(lambda: defaultdict(int))

for sequence in training_sequences:
    for current_state, next_state in zip(sequence[:-1], sequence[1:]):
        transition_counts[current_state][next_state] += 1

# Convert transition counts into probabilities
transition_probabilities = {}

for current_state, next_states in transition_counts.items():
    total = sum(next_states.values())

    transition_probabilities[current_state] = {
        next_state: count / total
        for next_state, count in next_states.items()
    }

# Hybrid sequence generation
def generate_sequence(start_state="START", max_steps=10):
    sequence = [start_state]
    current_state = start_state

    for _ in range(max_steps):
        # Restrict candidates to valid FSM transitions
        valid_next_states = fsm_transitions.get(current_state, [])

        if not valid_next_states:
            break

        # Obtain probabilities for valid transitions
        probabilities = [
            transition_probabilities.get(current_state, {}).get(state, 0)
            for state in valid_next_states
        ]

        # If probabilities are unavailable, use uniform distribution
        if sum(probabilities) == 0:
            probabilities = [1 / len(valid_next_states)] * len(valid_next_states)

        # Select next state probabilistically
        next_state = random.choices(
            valid_next_states,
            weights=probabilities,
            k=1
        )[0]

        sequence.append(next_state)
        current_state = next_state

        # Stop if terminal state is reached
        if current_state == "END":
            break

    return sequence

# Generate sequences
print("Generated Hybrid Sequences:\n")

for i in range(5):
    generated_sequence = generate_sequence()
    print(f"Sequence {i + 1}: {' -> '.join(generated_sequence)}")

# Validate a sequence using FSM
def validate_sequence(sequence):
    for current_state, next_state in zip(sequence[:-1], sequence[1:]):
        if next_state not in fsm_transitions.get(current_state, []):
            return False
    return sequence[-1] == "END"

# Test sequence validation
test_sequence = ["START", "GREETING", "QUESTION", "ANSWER", "END"]

print("\nTest Sequence:")
print(" -> ".join(test_sequence))

print("Valid Sequence:", validate_sequence(test_sequence))

Generated Hybrid Sequences:

Sequence 1: START -> GREETING -> END
Sequence 2: START -> GREETING -> END
Sequence 3: START -> GREETING -> END
Sequence 4: START -> GREETING -> QUESTION -> ANSWER -> END
Sequence 5: START -> GREETING -> QUESTION -> ANSWER -> END

Test Sequence:
START -> GREETING -> QUESTION -> ANSWER -> END
Valid Sequence: True
